# Extração de texto de um livro com o modelo OCR escolhido

Notebook do TCC *Reconhecimento de Texto em Imagens de Obras Literárias Antigas em Português*
(Ewaldo Uhlmann, UFSC, 2026). Aplica o pipeline de extração
([`src/pipeline/extractor.py`](https://github.com/Ewaldouhlmann/tcc-ocr-bu-ufsc/blob/main/src/pipeline/extractor.py))
a um livro em PDF guardado no seu Google Drive, usando um dos modelos avaliados no benchmark.

**Como usar**

1. **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4** (necessário para os VLMs locais; os
   modelos clássicos e os de API rodam sem GPU).
2. Suba o PDF do livro para o seu Google Drive.
3. Preencha o formulário da célula **1** (modelo, caminho do PDF, páginas).
4. **Ambiente de execução → Executar tudo.**

O texto de cada página é salvo no Drive assim que fica pronto. Se a sessão cair, rode tudo de novo: as páginas já
salvas são puladas.

## Qual modelo escolher

Resultados do benchmark do TCC (20 páginas transcritas à mão, Cap. 6). CER = taxa de erro por caractere; menor é melhor.

| Modelo (`MODELO`) | Tipo | Precisa de | CER médio | Observação |
|---|---|---|---|---|
| `qwen-7b` | VLM local (Ollama) | GPU | 1,6% | Melhor resultado entre os modelos abertos |
| `gemma4-e4b` | VLM local (Ollama) | GPU | 2,1% | |
| `glm-ocr` | VLM local (Ollama) | GPU recomendada | 5,8% | Melhor modelo que roda em CPU comum; repetiu texto em 1 página |
| `llamaparse-free` | VLM via API | chave `LLAMA_CLOUD_API_KEY` | 5,8% | Envia as imagens a um serviço externo |
| `doctr` | Clássico (CRNN) | — | 8,6% | |
| `easyocr` | Clássico (CRNN) | — | 14,8% | |
| `tesseract` | Clássico | — | 18,1% | Mais rápido e leve |
| `deepseek-ocr` | VLM local (Ollama) | GPU | 26,6% | Devolveu páginas vazias no benchmark |
| `claude` | VLM via API | chave `ANTHROPIC_API_KEY` (paga) | — | Fora do ranking (ver TCC, Cap. 6) |

Os valores são da amostra do benchmark; em outras obras o desempenho pode variar. Atualize esta tabela com
`benchmark/avaliar.py` se rodar novas avaliações.

In [ ]:
#@title 1. Configuração { display-mode: "form" }
#@markdown **Modelo** (ver tabela acima):
MODELO = "qwen-7b"  #@param ["qwen-7b", "gemma4-e4b", "glm-ocr", "deepseek-ocr", "llamaparse-free", "claude", "doctr", "easyocr", "tesseract"]
#@markdown **PDF do livro**, caminho dentro do *Meu Drive*:
PDF_NO_DRIVE = "livros/heloiza.pdf"  #@param {type:"string"}
#@markdown **Páginas** a processar, ex. `1-20,35`. Vazio = livro inteiro:
PAGINAS = ""  #@param {type:"string"}
#@markdown **Resolução** de renderização (DPI). 300 para digitalizações de boa qualidade; 150 basta para fotos de celular:
DPI = 300  #@param {type:"integer"}
#@markdown **Pasta de saída**, dentro do *Meu Drive*:
SAIDA_NO_DRIVE = "tcc_extracao"  #@param {type:"string"}
#@markdown Repositório do projeto:
REPO = "https://github.com/Ewaldouhlmann/tcc-ocr-bu-ufsc.git"  #@param {type:"string"}

OLLAMA = {"qwen-7b", "gemma4-e4b", "glm-ocr", "deepseek-ocr"}
print(f"Modelo: {MODELO} | PDF: {PDF_NO_DRIVE} | páginas: {PAGINAS or 'todas'} | DPI: {DPI}")

In [ ]:
#@title 2. Conecta o Google Drive e confere o PDF
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive")
PDF = DRIVE / PDF_NO_DRIVE
SAIDA = DRIVE / SAIDA_NO_DRIVE
assert PDF.exists(), f"PDF não encontrado: {PDF}. Confira o caminho na célula 1."
SAIDA.mkdir(parents=True, exist_ok=True)
print("PDF:", PDF)
print("Saída:", SAIDA / PDF.stem)

In [ ]:
#@title 3. Baixa o código do projeto (GitHub)
!rm -rf /content/tcc && git clone -q --depth 1 {REPO} /content/tcc
%cd /content/tcc
!git log -1 --format="Versão: %h (%cd)" --date=short

In [ ]:
#@title 4. Instala só o que o modelo escolhido precisa
!apt-get -qq install -y poppler-utils > /dev/null
!pip install -q pdf2image python-dotenv pyyaml

if MODELO == "tesseract":
    !apt-get -qq install -y tesseract-ocr tesseract-ocr-por > /dev/null
    !pip install -q pytesseract
elif MODELO == "easyocr":
    !pip install -q easyocr
elif MODELO == "doctr":
    !pip install -q "python-doctr[torch]"
elif MODELO == "llamaparse-free":
    !pip install -q llama-parse
elif MODELO == "claude":
    !pip install -q anthropic
elif MODELO in OLLAMA:
    !pip install -q ollama
print("Dependências instaladas para", MODELO)

In [ ]:
#@title 5. VLMs locais: instala o Ollama e baixa o modelo (pulado para os demais)
import subprocess, time, urllib.request, sys

if MODELO in OLLAMA:
    sys.path.insert(0, "/content/tcc")
    from src.ocr import MODEL_REGISTRY
    TAG = MODEL_REGISTRY[MODELO].model_tag

    !apt-get -qq install -y zstd > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh > /dev/null
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(30):
        try:
            urllib.request.urlopen("http://localhost:11434/api/version")
            break
        except Exception:
            time.sleep(2)
    print("Baixando", TAG, "(alguns minutos)...")
    !ollama pull {TAG} 2>&1 | tail -1
    !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "Sem GPU: o modelo vai rodar em CPU, bem mais devagar."
else:
    print("Modelo", MODELO, "não usa Ollama; nada a fazer aqui.")

In [ ]:
#@title 6. Modelos via API: lê a chave dos Secrets do Colab (pulado para os demais)
# Cadastre a chave no ícone de chave (🔑 Secrets) da barra lateral, com o nome indicado,
# e ative "Acesso do notebook". A chave nunca fica escrita no notebook.
import os
CHAVES = {"llamaparse-free": "LLAMA_CLOUD_API_KEY", "claude": "ANTHROPIC_API_KEY"}
if MODELO in CHAVES:
    from google.colab import userdata
    os.environ[CHAVES[MODELO]] = userdata.get(CHAVES[MODELO])
    print("Chave", CHAVES[MODELO], "carregada.")
else:
    print("Modelo", MODELO, "não usa API; nada a fazer aqui.")

In [ ]:
#@title 7. Extração (pode levar horas em livros longos; o progresso aparece abaixo)
args = f'--pdf "{PDF}" --model {MODELO} --output "{SAIDA}" --dpi {DPI}'
if PAGINAS.strip():
    args += f' --paginas "{PAGINAS.strip()}"'
!cd /content/tcc && python -m src.pipeline.extractor {args}

In [ ]:
#@title 8. Resultado: falhas e prévia do texto
import pandas as pd

pasta = SAIDA / PDF.stem
execucao = pd.read_csv(pasta / "execucao.csv")
print(f"{len(list((pasta / 'paginas').glob('*.txt')))} páginas salvas | "
      f"tempo mediano por página: {execucao['tempo_s'].median():.1f} s")
if (pasta / "falhas.csv").exists():
    print("Páginas com falha (rode a célula 7 de novo para tentar outra vez):")
    print((pasta / "falhas.csv").read_text())
livro = pasta / f"{PDF.stem}.txt"
print(f"Livro completo: {livro}\n")
print(livro.read_text(encoding="utf-8")[:2000])

## Arquivos gerados (no Drive, em `SAIDA_NO_DRIVE/<nome do PDF>/`)

| Arquivo | Conteúdo |
|---|---|
| `paginas/0001.txt`, ... | Texto de cada página, sem correção |
| `<nome do PDF>.txt` | Livro completo, com um marcador `===== Página N =====` antes de cada página |
| `execucao.csv` | Tempo, número de caracteres e tokens de cada página |
| `falhas.csv` | Páginas em que o modelo falhou (só existe se houve falha) |

**Limitações** (detalhadas no TCC): o texto sai sem revisão. VLMs podem repetir trechos em laço ou modernizar
grafias antigas; motores clássicos confundem caracteres parecidos (`ſ`→`f`, `o`→`0`). Para uso acadêmico, revise o
resultado contra as imagens.